In [1]:
# eduardosolanojaime
# 59ccdbb329db2aa86941c155afe85e0b

In [2]:
"""
!pip install opendatasets
import opendatasets as od
dataset_url = 'https://www.kaggle.com/competitions/predict-energy-behavior-of-prosumers/data'
od.download(dataset_url)
"""

"\n!pip install opendatasets\nimport opendatasets as od\ndataset_url = 'https://www.kaggle.com/competitions/predict-energy-behavior-of-prosumers/data'\nod.download(dataset_url)\n"

> Kristjn Eljand, Martin Laid, Jean-Baptiste Scellier, Sohier Dane, Maggie Demkin, Addison Howard. (2023). Enefit - Predict Energy Behavior of Prosumers. Kaggle. [Enefit - Predict Energy Behavior of Prosumers](https://www.kaggle.com/competitions/predict-energy-behavior-of-prosumers/overview)

## Importing libraries

In [19]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor, BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_squared_error

import warnings

## Setting options

In [4]:
warnings.simplefilter('ignore')
pd.set_option('display.max_columns', 100)
xgb.set_config(verbosity=0)

## Importing data

In [5]:
root = './Data'

data        = pd.read_csv(root + "/train.csv", low_memory=False)
client      = pd.read_csv(root + "/client.csv", low_memory=False)
gas         = pd.read_csv(root + "/gas_prices.csv", low_memory=False)
electricity = pd.read_csv(root + "/electricity_prices.csv", low_memory=False)
forecast    = pd.read_csv(root + "/forecast_weather.csv", low_memory=False)
historical  = pd.read_csv(root + "/historical_weather.csv", low_memory=False)
location    = pd.read_csv(root + "/weather_station_to_county_mapping.csv", low_memory=False)

## Data types manipulation

In [6]:
data['hour']        = pd.to_datetime(data['datetime']).dt.hour
data['date']        = pd.to_datetime(data['datetime']).dt.date
client['hour']      = pd.to_datetime(client['date']).dt.hour
client['date']      = pd.to_datetime(client['date']).dt.date
gas['hour']         = pd.to_datetime(gas['forecast_date']).dt.hour
gas['date']         = pd.to_datetime(gas['forecast_date']).dt.date
electricity['hour'] = pd.to_datetime(electricity['forecast_date']).dt.hour
electricity['date'] = pd.to_datetime(electricity['forecast_date']).dt.date
forecast['hour']    = pd.to_datetime(forecast['forecast_datetime']).dt.tz_convert('UTC').dt.hour
forecast['date']    = pd.to_datetime(forecast['forecast_datetime']).dt.tz_convert('UTC').dt.date
historical['hour']  = pd.to_datetime(historical['datetime']).dt.hour
historical['date']  = pd.to_datetime(historical['datetime']).dt.date
forecast['date']    = pd.to_datetime(forecast['forecast_datetime']).dt.tz_convert('UTC').dt.date

## Merge + other features

In [11]:
df = data.merge(client.drop('data_block_id', axis=1), on=['county', 'is_business', 'product_type', 'date', 'hour'], how="left")
df = df.merge(gas.loc[:, ['date', 'lowest_price_per_mwh', 'highest_price_per_mwh']], on='date', how='left')
df = df.merge(electricity.loc[:, ['euros_per_mwh', 'date', 'hour']], on=['date', 'hour'], how="left")

forecast_location   = forecast.merge(location, on=['longitude', 'latitude'], how="left")
forecast_location.drop(['forecast_datetime', 'origin_datetime', 'county_name', 'data_block_id'], axis=1, inplace=True)

temp = forecast_location.groupby(['date', 'county', 'hour']).mean()
df = df.merge(temp,  on=['date', 'county', 'hour'], how="left", suffixes=('', '_y'))


historical_location = historical.merge(location, on=['longitude', 'latitude'], how="left")
historical_location.drop(['datetime', 'longitude', 'latitude', 'county_name', 'data_block_id'], axis=1, inplace=True)

temp = historical_location.groupby(['hour', 'date', 'county']).mean()
df = df.merge(temp,  on=['date', 'hour', 'county'], how="left", suffixes=('', '_hist'))

df = df[df['target'].notnull()]

df['year'] = pd.to_datetime(data['datetime']).dt.year
df['month'] = pd.to_datetime(data['datetime']).dt.month
df['week'] = pd.to_datetime(data['datetime']).dt.isocalendar().week.astype(int)
df['day'] = pd.to_datetime(data['datetime']).dt.day
df['dayofweek'] = pd.to_datetime(data['datetime']).dt.dayofweek
df['monthofquarter'] = df['month']%4

estonian_holidays = [
    (1, 1),   # New Year's Day
    (2, 24),  # Independence Day
    (4, 15),  # Good Friday
    (4, 17),  # Easter Sunday
    (5, 1),   # Spring Day
    (5, 23),  # Whit Sunday
    (6, 23),  # Victory Day
    (6, 24),  # Midsummer Day
    (8, 20),  # Day of Restoration of Independence
    (12, 24), # Christmas Eve
    (12, 25), # Christmas Day
    (12, 26), # Boxing Day
]
def is_estonian_holiday(date):
  if date in estonian_holidays:
      return 1
  else:
      return 0

df['is_holiday'] = df.apply(lambda row: is_estonian_holiday((row['month'], row['day'])), axis=1)

df.drop(['datetime', 'date', 'data_block_id', 'prediction_unit_id', 'row_id', 'eic_count'], axis=1, inplace=True)

In [12]:
target = 'target'
numeric_cols = ['installed_capacity', 'lowest_price_per_mwh', 'highest_price_per_mwh', 'euros_per_mwh', 'latitude',
                'longitude', 'hours_ahead', 'temperature', 'dewpoint', 'cloudcover_high', 'cloudcover_mid', 'cloudcover_low',
                'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'direct_solar_radiation', 'snowfall',
                'surface_solar_radiation_downwards', 'total_precipitation', 'temperature_hist', 'dewpoint_hist', 'rain', 'snowfall_hist',
                'surface_pressure', 'cloudcover_high_hist', 'cloudcover_mid_hist', 'cloudcover_low_hist', 'cloudcover_total_hist',
                'windspeed_10m', 'winddirection_10m', 'shortwave_radiation', 'direct_solar_radiation_hist', 'diffuse_radiation']
categoric_cols = ['county', 'is_business', 'product_type', 'is_holiday', 'is_consumption']
time_cols = ['year', 'month', 'week', 'day', 'hour', 'monthofquarter', 'dayofweek']

sorted(list(df.columns)) == sorted(numeric_cols + categoric_cols + time_cols + [target])

True

## Data Transformation

In [13]:
target = 'target'
numeric_cols = ['installed_capacity', 'lowest_price_per_mwh', 'highest_price_per_mwh', 'euros_per_mwh', 'latitude',
                'longitude', 'hours_ahead', 'temperature', 'dewpoint', 'cloudcover_high', 'cloudcover_mid', 'cloudcover_low',
                'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'direct_solar_radiation', 'snowfall',
                'surface_solar_radiation_downwards', 'total_precipitation', 'temperature_hist', 'dewpoint_hist', 'rain', 'snowfall_hist',
                'surface_pressure', 'cloudcover_high_hist', 'cloudcover_mid_hist', 'cloudcover_low_hist', 'cloudcover_total_hist',
                'windspeed_10m', 'winddirection_10m', 'shortwave_radiation', 'direct_solar_radiation_hist', 'diffuse_radiation']
categoric_cols = ['county', 'is_business', 'product_type', 'is_holiday', 'is_consumption']
time_cols = ['year', 'month', 'week', 'day', 'hour', 'monthofquarter', 'dayofweek']

y = df['target']
X = df.drop(['target'], axis=1)

pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
ct = ColumnTransformer(transformers=
                       [('numeric', pipe, numeric_cols),
                        ('categoric', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categoric_cols)],
                       remainder='passthrough', verbose_feature_names_out=False)

temp = ct.fit_transform(X)
X_fit = pd.DataFrame(temp, columns=list(ct.get_feature_names_out()))
X_fit.rename(columns={'is_holiday_1': 'is_holiday', 'is_consumption_1': 'is_consumption'}, inplace=True)
df_fit = pd.concat([X_fit, y], axis=1)
df_fit = df_fit[df_fit['target']>0]

## Data Exploration and Analysis

It can be found in [my site](https://sites.google.com/up.edu.mx/prosumer-energy-behavior/home)

In [14]:
df_fit.sample(5)

,installed_capacity,lowest_price_per_mwh,highest_price_per_mwh,euros_per_mwh,latitude,longitude,hours_ahead,temperature,dewpoint,cloudcover_high,cloudcover_mid,cloudcover_low,cloudcover_total,10_metre_u_wind_component,10_metre_v_wind_component,direct_solar_radiation,snowfall,surface_solar_radiation_downwards,total_precipitation,temperature_hist,dewpoint_hist,rain,snowfall_hist,surface_pressure,cloudcover_high_hist,cloudcover_mid_hist,cloudcover_low_hist,cloudcover_total_hist,windspeed_10m,winddirection_10m,shortwave_radiation,direct_solar_radiation_hist,diffuse_radiation,county_0,county_1,county_2,county_3,county_4,county_5,county_6,county_7,county_8,county_9,county_10,county_11,county_12,county_13,county_14,county_15,is_business_1,product_type_0,product_type_1,product_type_2,product_type_3,is_holiday,is_consumption,hour,year,month,week,day,dayofweek,monthofquarter,target
1774777,-0.06446,-1.124173,-1.046818,-0.174042,0.386917,1.132180,-1.178769,-1.127925,-1.049700,2.016237,-0.599029,-1.332245,0.389134,0.118382,0.184907,2.590101,-0.164030,1.134317,-0.235248,-1.382981,-1.204350,-0.172696,-0.155909,0.438510,2.330737,0.189788,-1.102558,-0.288269,-0.920569,0.626582,-0.385982,-0.316219,-0.440418,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,7.0,2023.0,3.0,11.0,16.0,3.0,3.0,110.598
997968,-0.06446,1.357735,1.164432,0.028198,0.386917,0.239855,0.050295,-0.035261,-0.026016,-0.282378,-0.294473,-0.184539,0.386956,0.055051,0.024302,-0.384327,-0.164030,-0.383945,-0.221045,-0.024645,-0.031310,-0.172696,-0.155909,0.051420,-0.391421,-0.293182,-0.050523,0.249021,-0.074527,0.071334,-0.378509,-0.316219,-0.420059,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,23.0,2022.0,7.0,28.0,17.0,6.0,3.0,10.822
1481296,-0.06446,1.000172,0.875691,2.948965,0.386917,-0.206308,0.255139,-1.521767,-1.341471,1.960217,2.151072,1.227364,0.682698,0.899836,2.746677,-0.384387,3.738410,-0.383417,1.524405,-2.294163,-2.165858,-0.172696,-0.155909,-1.726951,-0.743700,-0.740375,0.756037,-0.140051,-0.034239,0.740898,0.309063,0.094690,0.658938,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,14.0,2022.0,12.0,50.0,14.0,2.0,0.0,10.359
334374,-0.06446,-0.135617,0.056797,2.495740,0.386917,0.239855,0.050295,-0.035261,-0.026016,-0.282378,-0.294473,-0.184539,0.386956,0.055051,0.024302,-0.384327,-0.164030,-0.383945,-0.221045,-0.024645,-0.031310,-0.172696,-0.155909,0.051420,-0.391421,-0.293182,-0.050523,0.249021,-0.074527,0.071334,-0.378509,-0.316219,-0.420059,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,13.0,2021.0,12.0,51.0,21.0,1.0,0.0,12.268
562981,-0.06446,-0.156650,0.762387,0.199343,0.386917,-0.206308,-0.154549,-1.074447,-1.407371,-1.013159,-0.974816,1.917488,0.575225,-0.786045,-2.493514,0.145277,0.157836,1.163831,-0.095075,-1.214244,-1.377421,-0.172696,-0.155909,0.573150,-0.743700,-0.955028,1.282054,0.230494,0.489502,-2.035345,1.370315,0.452918,2.857650,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,12.0,2022.0,3.0,9.0,4.0,4.0,3.0,1724.839


In [ ]:
df_fit.shape

(1666328, 64)

In [15]:
pca = PCA(len(X_fit.columns))
pca.fit(X_fit)
X_pca= pca.transform(X_fit)

for i in range(15):
  print(f"PC {i+1}: {round(pca.explained_variance_ratio_[:i].sum()*100, 2)}%")

PC 1: 0.0%
PC 2: 61.81%
PC 3: 79.54%
PC 4: 90.69%
PC 5: 92.23%
PC 6: 93.26%
PC 7: 94.19%
PC 8: 94.84%
PC 9: 95.33%
PC 10: 95.69%
PC 11: 96.04%
PC 12: 96.37%
PC 13: 96.68%
PC 14: 96.95%
PC 15: 97.2%


For complex models, the reduced dataset with the 15 principal components (amounting to 97.17% of the variance) will be used to lighten the computation time.

In [16]:
pca = PCA(n_components=15)
pca.fit(X_fit)
X_pca = pd.DataFrame(pca.transform(X_fit))
df_light = pd.concat([X_pca, y], axis=1)
df_light.sample(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,target
1517553,25.835908,8.062795,-9.485750,-2.935963,-3.152958,2.678911,2.523313,0.267217,-1.187387,0.125038,-1.634518,-0.228625,-0.969364,-0.281929,-0.174863,707.170
946411,-0.620542,-14.663561,-4.793706,7.985793,4.449676,1.605209,-0.969776,-0.510954,-1.957326,-0.501938,-0.912914,-0.912786,1.700131,-1.406686,0.609856,589.667
671583,-12.737839,-8.086940,-1.611128,-0.886763,5.322160,1.371654,0.139856,-1.247627,1.262031,-0.240732,-4.054353,-1.538102,-1.863352,0.363999,0.090599,619.973
702506,-11.305112,1.825671,10.492525,-0.074703,-1.424881,2.718212,-1.063703,0.484904,-0.015738,-0.173982,-0.363198,0.309487,-1.488394,-0.415308,0.083706,0.000
1021608,4.399688,9.097525,1.512165,-0.287797,-0.096246,-3.227195,-0.960316,0.709098,0.524396,-0.127076,-0.156452,-0.636903,1.515850,0.363478,0.307458,9.372


## Data Splitting

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X_fit, y, test_size=1/3)
l_X_train, l_X_test, l_y_train, l_y_test = train_test_split(X_pca, y, test_size=1/3)

## Short list of dirty models

### xgboost

In [20]:
model = xgb.XGBRegressor()
model.fit(X_train, y_train)
score = mean_squared_error(model.predict(X_train), y_train)
print(f"Score of {score:.5f} was achieved with {model}")

Score of 34972.77427 was achieved with XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)


In [21]:
model = MLPRegressor()
model.fit(l_X_train, l_y_train)
score = mean_squared_error(model.predict(l_X_train), l_y_train)
print(f"Score of {score:.5f} was achieved with {model}")

Score of 808700.07174 was achieved with MLPRegressor()
